# YOLO Pipeline Example

This notebook demonstrates how to download two datasets (e.g., COCO8 and COCO128 - yielding > 100 images total), create a YAML configuration file for merging and training, and utilize the Sentry scripts to build the dataset and train a YOLO model. We use the YAML config to steer both the merging behavior and the training loop.

In [23]:
import json

In [13]:
import os
import urllib.request
import zipfile
from pathlib import Path

# 1. Setup raw data directory
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# 2. Download and unpack two datasets (~136 images total)
datasets = {
    "coco8": "https://github.com/ultralytics/yolov5/releases/download/v1.0/coco8.zip",
    "coco128": "https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip"
}

for name, url in datasets.items():
    zip_path = raw_dir / f"{name}.zip"
    extracted_path = raw_dir / name
    
    if not extracted_path.exists():
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, zip_path)
        print(f"Extracting {name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(raw_dir)
        print(f"{name} ready.\n")
    else:
        print(f"{name} already exists.\n")

Extracting coco8...
coco8 ready.

Extracting coco128...
coco128 ready.



## Configuration
We actively use YAML files to manage the configuration. We will define both datasets, merging parameters, and training settings. We save this directly to `./configs/pipeline_config.yaml`.

In [ ]:
yaml_config = """project:
  name: "demo_yolo"
  runs_dir: "runs/"

dataset:
  # Our two downloaded datasets
  dataset_1: "data/raw/coco8/"
  dataset_2: "data/raw/coco128/"
  merged_dir: "data/processed/merged_coco/"
  
  merge_mode: "rebuild"
  rebuild_ratios:
    train: 0.7
    val: 0.2
    test: 0.1

training:
  model_yaml: "yolo26n.pt"
  epochs: 10
  batch_size: 8
  imgsz: 320
  device: "cpu"
  workers: 1
  lr0: 0.001
"""

config_dir = Path("configs")
config_dir.mkdir(exist_ok=True)

with open(config_dir / "pipeline_config.yaml", "w") as f:
    f.write(yaml_config)
    
print("Created pipeline_config.yaml")

Created pipeline_config.yaml


## Validation and Auditing
Let's check if the downloaded datasets conform to our YOLO standards using the provided CLI tools.

In [18]:
# Validating COCO8 Dataset
from sentry_ai.dataset.validate import validate_yolo_dataset

def validate_dataset(dataset_path: str) -> None:
    errors = validate_yolo_dataset(dataset_path)

    if errors:
        print(f"[-] Dataset Validation FAILED with {len(errors)} errors:")

        for i, error in enumerate(errors[:20]):
            print(f"{i+1}. {error}")

        print("\n... (showing first 20 errors)...")

    else:
        print(f"[+] Validation of {dataset_path} dataset passed.")

In [19]:
validate_dataset("data/raw/coco8")

[+] Validation of data/raw/coco8 dataset passed.


In [20]:
validate_dataset("data/raw/coco128")

[-] Dataset Validation FAILED with 4 errors:
1. Missing pair: Image 000000000250.jpg has no corresponding label file.
2. Missing pair: Image 000000000508.jpg has no corresponding label file.
3. Missing pair: Label 000000000656.txt has no corresponding image file.
4. Missing pair: Label 000000000659.txt has no corresponding image file.

... (showing first 20 errors)...


In [24]:
from sentry_ai.dataset.audit import audit_yolo_dataset

def audit_dataset(dataset_path: str) -> None:
    stats = audit_yolo_dataset(dataset_path)
    
    print(f"Dataset Audit Statistics for {dataset_path}:")
    print(json.dumps(stats, indent=4))

In [26]:
audit_dataset("data/raw/coco8")

Dataset Audit Statistics for data/raw/coco8:
{
    "total_images": 8,
    "labeled_images": 8,
    "empty_label_images": 0,
    "total_bboxes": 30,
    "class_distribution": {
        "45": 3,
        "50": 1,
        "49": 4,
        "23": 2,
        "58": 2,
        "75": 1,
        "22": 1,
        "25": 1,
        "0": 10,
        "16": 1,
        "17": 2,
        "20": 2
    },
    "bbox_size_percentiles": {
        "min": 0.0007163036440000001,
        "25th": 0.009477320108985,
        "median": 0.025956492369000002,
        "75th": 0.248637001375,
        "max": 0.639821636596
    }
}


In [27]:
audit_dataset("data/raw/coco128")

Dataset Audit Statistics for data/raw/coco128:
{
    "total_images": 128,
    "labeled_images": 126,
    "empty_label_images": 2,
    "total_bboxes": 929,
    "class_distribution": {
        "45": 28,
        "50": 11,
        "49": 4,
        "23": 9,
        "58": 14,
        "75": 2,
        "22": 4,
        "25": 18,
        "0": 254,
        "16": 9,
        "17": 2,
        "20": 17,
        "2": 46,
        "7": 12,
        "11": 2,
        "74": 9,
        "6": 3,
        "3": 5,
        "1": 6,
        "36": 5,
        "4": 6,
        "26": 19,
        "43": 16,
        "69": 5,
        "68": 3,
        "73": 29,
        "42": 6,
        "55": 4,
        "13": 9,
        "56": 35,
        "53": 5,
        "60": 13,
        "41": 36,
        "44": 22,
        "59": 3,
        "77": 21,
        "72": 5,
        "71": 6,
        "39": 18,
        "46": 1,
        "48": 2,
        "14": 16,
        "33": 10,
        "40": 16,
        "27": 7,
        "76": 1,
        "34": 4,
    

## Merging
Merge them using our configuration file. This will optionally remap IDs (if we had configured `class_remap`), combine the two distinct collections into a new directory, and shuffle data into train/val subsets as configured.

In [ ]:
from sentry_ai.dataset.merge import merge_datasets
from sentry_ai.config import load_config

config = load_config("configs/pipeline_config.yaml")
merge_datasets(config)

In [ ]:
audit_dataset(r"data/processed/merged_coco")

## Train
Finally, initiate training on the merged configurations based strictly heavily on `pipeline_config.yaml` values.

In [34]:
from sentry_ai.yolo.train import train_yolo

train_yolo(config)


Initializing YOLO with yolo26n.pt...
Starting training...
New https://pypi.org/project/ultralytics/8.4.18 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.9 torch-2.10.0+cpu CPU (13th Gen Intel Core(TM) i7-13700H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\projects\Sentry\notebooks\data\processed\merged_coco\sentry_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300